# 03 Advanced Analysis

This notebook covers anomaly detection, feature importance, air quality correlation, and spatial weather patterns.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

import plotly.express as px

from data_processing import load_weather_data, clean_weather_data, get_temperature_column, add_anomaly_labels
from visualization import spatial_temperature_figure

In [ ]:
raw_path = PROJECT_ROOT / 'data' / 'raw' / 'GlobalWeatherRepository.csv'
df = clean_weather_data(load_weather_data(raw_path))
temp_col = get_temperature_column(df)
df.head()

In [ ]:
features = [col for col in [temp_col, 'humidity', 'wind_kph', 'precip_mm', 'pressure_mb'] if col in df.columns]
anomaly_df = add_anomaly_labels(df, features)
anomaly_df['anomaly'].value_counts()

In [ ]:
sample = anomaly_df.sample(min(len(anomaly_df), 5000), random_state=42)
px.scatter(sample, x='last_updated', y=temp_col, color='anomaly', title='Temperature Anomaly Detection')

In [ ]:
air_quality_cols = [col for col in df.columns if 'air_quality' in col.lower() or 'pm2' in col.lower()]
weather_cols = [col for col in [temp_col, 'humidity', 'wind_kph', 'pressure_mb', 'precip_mm'] if col in df.columns]
corr_cols = air_quality_cols + weather_cols
if corr_cols:
    px.imshow(df[corr_cols].corr(numeric_only=True), color_continuous_scale='RdBu_r', zmin=-1, zmax=1, title='Air Quality and Weather Correlation')

In [ ]:
spatial_temperature_figure(df, temp_col)